### Cell 1: Import thư viện và các module đã tách

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Import từ các file của bạn
from config import *
from data_utils import *
from model_utils import *

### Cell 2: Đọc và Xử lý dữ liệu (ETL)

In [ ]:
df_fake = pd.read_csv(RAW_FAKE_PATH)
df_fake['label'] = 1
df_true = pd.read_csv(RAW_TRUE_PATH)
df_true['label'] = 0

explore_raw_data(df_fake, "TẬP TIN GIẢ (FAKE.CSV)")
explore_raw_data(df_true, "TẬP TIN THẬT (TRUE.CSV)")

df_fake['date_parsed'] = pd.to_datetime(df_fake['date'], errors='coerce')
df_true['date_parsed'] = pd.to_datetime(df_true['date'], errors='coerce')

df_fake = df_fake.dropna(subset=['text', 'date_parsed']).drop_duplicates(subset=['text'])
df_true = df_true.dropna(subset=['text', 'date_parsed']).drop_duplicates(subset=['text'])

df_fake = df_fake.sort_values('date_parsed').reset_index(drop=True)
df_true = df_true.sort_values('date_parsed').reset_index(drop=True)

df_fake['is_test'] = False
df_fake.loc[int(len(df_fake) * 0.8):, 'is_test'] = True

df_true['is_test'] = False
df_true.loc[int(len(df_true) * 0.8):, 'is_test'] = True


df = pd.concat([df_fake, df_true], axis=0)
df = df.sort_values('is_test').reset_index(drop=True)

df['text_clean_raw'] = df['text'].apply(deep_clean_text)
df['title_text_clean_raw'] = (df['title'] + " " + df['text']).apply(deep_clean_text)
df['title_clean_raw'] = df['title'].apply(deep_clean_text)

df['text_clean'] = df['text_clean_raw'].apply(lemmatize_and_remove_stopwords)
df['title_text_clean'] = df['title_text_clean_raw'].apply(lemmatize_and_remove_stopwords)
df['title_clean'] = df['title_clean_raw'].apply(lemmatize_and_remove_stopwords)

# Xóa trùng lặp sâu
df = df.drop_duplicates(subset=['text_clean'], keep='first').reset_index(drop=True)
df = df[df['text_clean'].str.strip() != ""].reset_index(drop=True)

train_idx = df[df['is_test'] == False].index
test_idx = df[df['is_test'] == True].index
y_train = df.loc[train_idx, 'label']
y_test = df.loc[test_idx, 'label']

df.drop(columns=['subject', 'date', 'date_parsed', 'is_test'], inplace=True, errors='ignore')

explore_processed_data(df, train_idx, test_idx)

test_idx = remove_data_leakage_from_test(df, train_idx, test_idx, clean_col='text_clean')
y_test = df.loc[test_idx, 'label']

processed_dir = os.path.dirname(PROCESSED_DATA_PATH)
os.makedirs(processed_dir, exist_ok=True)
print(f"\n⏳ Đang lưu tập dữ liệu đã xử lý ra file CSV (có thể mất vài phút)...")
df.to_csv(PROCESSED_DATA_PATH, index=False, encoding='utf-8')
print(f"✅ Đã lưu thành công tại: {PROCESSED_DATA_PATH}")
joblib.dump(train_idx, os.path.join(processed_dir, 'train_idx.pkl'))
joblib.dump(test_idx, os.path.join(processed_dir, 'test_idx.pkl'))
print(f"✅ Đã lưu file chia index (train_idx, test_idx) thành công!")

### Cell 3: Phân tích EDA

In [ ]:
plot_eda(df)

### Cell 4: Xây dựng Feature Pipeline

In [ ]:
X_train_text, X_test_text, tfidf_text = build_feature_pipeline(df, train_idx, test_idx, 'text_clean', 'text_clean_raw')
X_train_title_text, X_test_title_text, tfidf_title_text = build_feature_pipeline(df, train_idx, test_idx, 'title_text_clean', 'title_text_clean_raw')
X_train_title, X_test_title, tfidf_title = build_feature_pipeline(df, train_idx, test_idx, 'title_clean', 'title_clean_raw')

### Cell 5: Huấn luyện Mô hình

In [ ]:
all_results = []

# ==========================================
# 1. LOGISTIC REGRESSION
# ==========================================
print("--- HUẤN LUYỆN LOGISTIC REGRESSION ---")
lr_model_text = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model_title_text = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model_title = LogisticRegression(max_iter=1000, class_weight='balanced')

acc_lr_text, rep_lr_text = train_and_evaluate_model(lr_model_text, "LR (Text Only)", X_train_text, y_train, X_test_text, y_test, "Blues")
acc_lr_title_text, rep_lr_title_text = train_and_evaluate_model(lr_model_title_text, "LR (Title + Text)", X_train_title_text, y_train, X_test_title_text, y_test, "Blues")
acc_lr_title, rep_lr_title = train_and_evaluate_model(lr_model_title, "LR (Title Only)", X_train_title, y_train, X_test_title, y_test, "Blues")

display_comparison_report("LOGISTIC REGRESSION", acc_lr_title_text, rep_lr_title_text, acc_lr_text, rep_lr_text, acc_lr_title, rep_lr_title)

all_results.extend([
    ("Logistic Regression", "Text Only", acc_lr_text, rep_lr_text),
    ("Logistic Regression", "Title + Text", acc_lr_title_text, rep_lr_title_text),
    ("Logistic Regression", "Title Only", acc_lr_title, rep_lr_title)
])

# ==========================================
# 2. NAIVE BAYES
# ==========================================
print("--- HUẤN LUYỆN NAIVE BAYES ---")
nb_model_text = MultinomialNB()
nb_model_title_text = MultinomialNB()
nb_model_title = MultinomialNB()

acc_nb_text, rep_nb_text = train_and_evaluate_model(nb_model_text, "Naive Bayes (Text Only)", X_train_text, y_train, X_test_text, y_test, "Greens")
acc_nb_title_text, rep_nb_title_text = train_and_evaluate_model(nb_model_title_text, "Naive Bayes (Title + Text)", X_train_title_text, y_train, X_test_title_text, y_test, "Greens")
acc_nb_title, rep_nb_title = train_and_evaluate_model(nb_model_title, "Naive Bayes (Title Only)", X_train_title, y_train, X_test_title, y_test, "Greens")

display_comparison_report("NAIVE BAYES", acc_nb_title_text, rep_nb_title_text, acc_nb_text, rep_nb_text, acc_nb_title, rep_nb_title)

all_results.extend([
    ("Naive Bayes", "Text Only", acc_nb_text, rep_nb_text),
    ("Naive Bayes", "Title + Text", acc_nb_title_text, rep_nb_title_text),
    ("Naive Bayes", "Title Only", acc_nb_title, rep_nb_title)
])

# ==========================================
# 3. SVM
# ==========================================
print("--- HUẤN LUYỆN SVM ---")
svm_model_text = LinearSVC(random_state=RANDOM_STATE, max_iter=2000, class_weight='balanced')
svm_model_title_text = LinearSVC(random_state=RANDOM_STATE, max_iter=2000, class_weight='balanced')
svm_model_title = LinearSVC(random_state=RANDOM_STATE, max_iter=2000, class_weight='balanced')

acc_svm_text, rep_svm_text = train_and_evaluate_model(svm_model_text, "SVM (Text Only)", X_train_text, y_train, X_test_text, y_test, "Oranges")
acc_svm_title_text, rep_svm_title_text = train_and_evaluate_model(svm_model_title_text, "SVM (Title + Text)", X_train_title_text, y_train, X_test_title_text, y_test, "Oranges")
acc_svm_title, rep_svm_title = train_and_evaluate_model(svm_model_title, "SVM (Title Only)", X_train_title, y_train, X_test_title, y_test, "Oranges")

display_comparison_report("SVM", acc_svm_title_text, rep_svm_title_text, acc_svm_text, rep_svm_text, acc_svm_title, rep_svm_title)

all_results.extend([
    ("SVM", "Text Only", acc_svm_text, rep_svm_text),
    ("SVM", "Title + Text", acc_svm_title_text, rep_svm_title_text),
    ("SVM", "Title Only", acc_svm_title, rep_svm_title)
])

### Cell 6: Tổng hợp metric

In [ ]:
summary_df = generate_metrics_summary(all_results, METRICS_SUMMARY_PATH)

### Cell 7: WordCloud

In [ ]:
plot_wordclouds(df)

### Cell 8: xAI

In [ ]:
import importlib
import xai_utils
import model_utils
importlib.reload(xai_utils)
importlib.reload(model_utils)
import numpy as np
import pandas as pd
import textwrap
from sklearn.linear_model import LogisticRegression

print("======================================================")
print("0. ĐÁNH GIÁ TRỌNG SỐ TỪ KHÓA TỪ MÔ HÌNH CHÍNH (GLOBAL IMPORTANCE)")
print("======================================================")
model_utils.plot_top_features(tfidf_title_text, lr_model_title_text, "Logistic Regression (Title + Text)")

print("\n======================================================")
print("1. CHUẨN BỊ MÔ HÌNH THUẦN CHỮ CHO xAI (TRÁNH LỖI META FEATURES)")
print("======================================================")

X_train_title_text_strings = df.loc[train_idx, 'title_text_clean'].values
y_train = df.loc[train_idx, 'label'].values

X_train_pure_tfidf = tfidf_title_text.transform(X_train_title_text_strings)

lr_xai_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_xai_model.fit(X_train_pure_tfidf, y_train)

print("\n======================================================")
print("2. TRÍCH XUẤT LỖI TỪ MÔ HÌNH GỐC VÀ CHẠY LIME RA CSV")
print("======================================================")

# LẤY DỰ ĐOÁN TỪ MÔ HÌNH GỐC (Mô hình có 5003 features)
y_pred_main = lr_model_title_text.predict(X_test_title_text) 
y_test = df.loc[test_idx, 'label'].values

top10_fp, top10_fn = model_utils.analyze_errors(df, test_idx, y_test, y_pred_main)
top20_actual_errors = pd.concat([top10_fp, top10_fn])

if len(top20_actual_errors) > 0:
    csv_path = '../results/lime_error_analysis_top20.csv'
    
    # Dùng mô hình thuần chữ (lr_xai_model) làm Proxy để bóc tách từ khóa cho các mẫu lỗi này
    lime_report_df = xai_utils.export_batch_lime_to_csv(top20_actual_errors, lr_xai_model, tfidf_title_text, csv_path)
    
    print("\n[XEM TRƯỚC BẢNG KẾT QUẢ XUẤT CSV CHI TIẾT]")
    display(lime_report_df[['Nhãn_Thực_Tế', 'Mô_Hình_Đoán', 'Từ_Khóa_Kéo_Về_Fake', 'Từ_Khóa_Kéo_Về_True']].head(5))
else:
    print("Không có lỗi nào để phân tích.")

print("\n======================================================")
print("3. ĐÁNH GIÁ TỔNG THỂ TỪ KHÓA BẰNG SHAP SUMMARY PLOT")
print("======================================================")
X_sample_text_strings = df.loc[test_idx, 'title_text_clean'].sample(n=200, random_state=42)
xai_utils.explain_with_shap(lr_xai_model, tfidf_title_text, X_sample_text_strings)

### Cell 9: Lưu mô hình

In [ ]:
import joblib
import os

# Tạo thư mục lưu mô hình (nếu chưa có)
os.makedirs('../models', exist_ok=True)

# Lưu bộ mã hóa TF-IDF và mô hình Logistic Regression
joblib.dump(tfidf_title_text, '../models/tfidf_title_text.pkl')
joblib.dump(lr_xai_model, '../models/lr_xai_model.pkl')

print("✅ Đã xuất mô hình và TF-IDF thành công vào thư mục ../models!")